# E-12 · Selecao de modelo: algoritmos, objetivo de ranking, comparaveis e Optuna

Protocolo **pre-registrado** em `docs/07` (escrito antes de rodar). Referencia: **CB_EMB** do E-08 (CatBoost, alvo = desvio sobre a mediana do canal, perda MAE, embedding SigLIP 2 da thumbnail com PCA 16), parametros fixos.

| Braco | O que muda |
| --- | --- |
| `LGBM` | LightGBM, perda L1, busca aleatoria (10) |
| `XGB` | XGBoost, perda absoluta, busca aleatoria (10) |
| `ET` | ExtraTrees, busca aleatoria (6) |
| `ENS` | media do CB_EMB com uma regressao linear robusta (LAD) com embedding |
| `CB_RANK` | CatBoost **YetiRank** agrupado por canal: a ordem dentro do canal vem do ranking, o nivel vem do CB_EMB |
| `CB_KNN` | CB_EMB + desvio medio dos 5 Shorts **anteriores** mais parecidos do mesmo canal + novidade visual |
| `CB_W` | CB_EMB com peso igual por canal |

**Regra:** substitui o CB_EMB so com p < 0,05 apos Holm (7 candidatos); `CB_RANK` e julgado pela acuracia por pares. **Fase 2:** Optuna (40 tentativas) nas origens 1-2, avaliado so na origem 3. PyCaret: exploratorio.

**Entrada:** arraste `e12_bundle.tar.gz` (pasta `case_galaxies/.tmp_transfer/`) para o painel Arquivos e rode tudo. CPU basta (~25-35 min).

In [ ]:
import os, sys, json, time, tarfile, itertools, subprocess, importlib, warnings
warnings.filterwarnings("ignore")
for pkg in ("catboost", "lightgbm", "xgboost", "optuna"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy as np, pandas as pd
from catboost import CatBoostRegressor, CatBoostRanker, Pool
import lightgbm as lgb, xgboost as xgb, optuna
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import spearmanr

SEED = 42
N_TRIALS = int(os.environ.get("E12_TRIALS", 10))
OPTUNA_TRIALS = int(os.environ.get("E12_OPTUNA", 40))
WORK = os.environ.get("E12_WORK", "/content/e12")
OUT = os.path.join(WORK, "outputs"); os.makedirs(OUT, exist_ok=True)
bundle = next((p for p in ("/content/e12_bundle.tar.gz", os.path.join(WORK, "e12_bundle.tar.gz")) if os.path.exists(p)), None)
if not os.path.exists(os.path.join(WORK, "bundle", "training_snapshot.csv")):
    assert bundle, "arraste e12_bundle.tar.gz para o painel Arquivos"
    with tarfile.open(bundle) as tar:
        tar.extractall(WORK)
B = os.path.join(WORK, "bundle")
gold = pd.read_csv(os.path.join(B, "training_snapshot.csv"), parse_dates=["published_at"])
z = np.load(os.path.join(B, "thumbnail_embeddings.npz"), allow_pickle=True)
EMB = dict(zip(z["video_id"].astype(str), z["embeddings"].astype(np.float32)))
CB_PARAMS = json.load(open(os.path.join(B, "e08_run_info.json")))["final_params"]
CB_PARAMS = {k: (int(v) if k in ("depth", "iterations") else v) for k, v in CB_PARAMS.items()}
TARGET, BASELINE = "target_log_likes", "baseline_log_likes"
data = gold[gold["split"] == "train"].sort_values(["channel_id", "published_at", "video_id"]).reset_index(drop=True)
data["resid"] = data[TARGET] - data[BASELINE]
E = np.vstack([EMB[v] for v in data["video_id"]])
print(len(data), "linhas,", data["channel_id"].nunique(), "canais | CB_EMB:", CB_PARAMS)

## Features de comparaveis (so Shorts anteriores do mesmo canal)

In [ ]:
K = 5
knn_mean, knn_sim, novelty = np.full(len(data), np.nan), np.full(len(data), np.nan), np.full(len(data), np.nan)
for _, idx in data.groupby("channel_id").groups.items():
    idx = np.asarray(idx)
    t = data.loc[idx, "published_at"].to_numpy()
    for j, i in enumerate(idx):
        prior = idx[t < t[j]]                       # estritamente antes: sem vazamento
        if len(prior) < K:
            continue
        sims = E[prior] @ E[i]
        top = np.argsort(-sims)[:K]
        knn_mean[i] = data.loc[prior[top], "resid"].mean()
        knn_sim[i] = sims[top].mean()
        centroid = E[prior].mean(0)
        novelty[i] = 1 - float(centroid @ E[i]) / (np.linalg.norm(centroid) + 1e-9)
data["knn5_resid_mean"], data["knn5_sim_mean"], data["visual_novelty"] = knn_mean, knn_sim, novelty
KNN_COLS = ["knn5_resid_mean", "knn5_sim_mean", "visual_novelty"]
print(data[KNN_COLS].describe().round(3))

## Protocolo (igual ao E-08)

In [ ]:
BASE_NUM = ["duration_seconds", "channel_prior_posts", "baseline_log_likes", "days_since_previous_post",
            "title_length", "description_length", "tag_count"]
BASE_CAT = ["channel_id", "sector", "channel_type", "publish_hour_utc", "publish_weekday"]
EMB_COLS = [f"emb_pc{i:02d}" for i in range(16)]
FRACTIONS = (0.50, 0.65, 0.80)

def folds(frame):
    out = []
    for pos, frac in enumerate(FRACTIONS):
        nxt = FRACTIONS[pos + 1] if pos + 1 < len(FRACTIONS) else 1.0
        fit, val = [], []
        for _, ch in frame.groupby("channel_id"):
            end = max(5, int(np.floor(len(ch) * frac))); stop = min(max(end + 1, int(np.floor(len(ch) * nxt))), len(ch))
            fit.extend(ch.index[:end]); val.extend(ch.index[end:stop])
        out.append((f"origin_{pos + 1}", np.array(fit), np.array(val)))
    return out

def holdout(idx):
    sub = data.loc[idx]
    dev, hold = [], []
    for _, ch in sub.groupby("channel_id"):
        n = max(1, int(np.ceil(len(ch) * 0.2))); dev.extend(ch.index[:-n]); hold.extend(ch.index[-n:])
    return np.array(dev), np.array(hold)

def with_pcs(fit_idx, idx_list):
    mean = E[fit_idx].mean(0); _, _, vt = np.linalg.svd(E[fit_idx] - mean, full_matrices=False); comp = vt[:16]
    frames = []
    for idx in idx_list:
        f = data.loc[idx].copy(); f[EMB_COLS] = (E[idx] - mean) @ comp.T; frames.append(f)
    return frames

def macro_mae(frame, pred):
    return float(pd.Series(np.abs(frame[TARGET].to_numpy() - pred), index=frame.index).groupby(frame["channel_id"]).mean().mean())

def cb_pool(frame, num, label=True, weight=None):
    x = frame[num + BASE_CAT].copy()
    for c in BASE_CAT: x[c] = x[c].astype(str)
    return Pool(x, frame["resid"].to_numpy() if label else None, cat_features=BASE_CAT, weight=weight)

def cb_fit_predict(fit, val, num, params=CB_PARAMS, weight=None):
    m = CatBoostRegressor(loss_function="MAE", random_seed=SEED, verbose=False, allow_writing_files=False, thread_count=-1, **params)
    m.fit(cb_pool(fit, num, weight=weight))
    return val[BASELINE].to_numpy() + m.predict(cb_pool(val, num, label=False))

def as_category(fit, others, cols):
    out = []
    for f in [fit] + others:
        f = f.copy()
        for c in cols:
            f[c] = pd.Categorical(f[c].astype(str), categories=sorted(fit[c].astype(str).unique()))
        out.append(f)
    return out

## Bracos

In [ ]:
rng = np.random.default_rng(SEED)
def sample(space, n):
    keys, seen, out = list(space), set(), []
    while len(out) < min(n, int(np.prod([len(v) for v in space.values()]))):
        p = {k: space[k][rng.integers(len(space[k]))] for k in keys}
        if tuple(p.values()) not in seen: seen.add(tuple(p.values())); out.append(p)
    return out

LGB_SPACE = {"num_leaves": [7, 15, 31], "learning_rate": [0.01, 0.03, 0.05], "min_child_samples": [10, 20, 40],
             "feature_fraction": [0.6, 0.8, 1.0], "bagging_fraction": [0.7, 0.9, 1.0], "lambda_l2": [0, 1, 5]}
def lgbm_arm(fit, val, dev, hold, num):
    best = None
    for p in sample(LGB_SPACE, N_TRIALS):
        d, h = as_category(dev, [hold], BASE_CAT)
        m = lgb.LGBMRegressor(objective="l1", n_estimators=3000, bagging_freq=1, random_state=SEED, verbose=-1, **p)
        m.fit(d[num + BASE_CAT], d["resid"], eval_set=[(h[num + BASE_CAT], h["resid"])], callbacks=[lgb.early_stopping(150, verbose=False)])
        s = macro_mae(hold, hold[BASELINE].to_numpy() + m.predict(h[num + BASE_CAT]))
        if best is None or s < best[0]: best = (s, p, max(50, m.best_iteration_ or 50))
    f, v = as_category(fit, [val], BASE_CAT)
    m = lgb.LGBMRegressor(objective="l1", n_estimators=int(best[2] * 1.1), bagging_freq=1, random_state=SEED, verbose=-1, **best[1])
    m.fit(f[num + BASE_CAT], f["resid"])
    return val[BASELINE].to_numpy() + m.predict(v[num + BASE_CAT]), best[1]

XGB_SPACE = {"max_depth": [3, 4, 6], "learning_rate": [0.01, 0.03, 0.05], "min_child_weight": [1, 5, 10],
             "subsample": [0.7, 0.9, 1.0], "colsample_bytree": [0.6, 0.8, 1.0], "reg_lambda": [1, 5]}
def xgb_arm(fit, val, dev, hold, num):
    best = None
    for p in sample(XGB_SPACE, N_TRIALS):
        d, h = as_category(dev, [hold], BASE_CAT)
        m = xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=3000, tree_method="hist", enable_categorical=True,
                             early_stopping_rounds=150, random_state=SEED, **p)
        m.fit(d[num + BASE_CAT], d["resid"], eval_set=[(h[num + BASE_CAT], h["resid"])], verbose=False)
        s = macro_mae(hold, hold[BASELINE].to_numpy() + m.predict(h[num + BASE_CAT]))
        if best is None or s < best[0]: best = (s, p, max(50, m.best_iteration + 1))
    f, v = as_category(fit, [val], BASE_CAT)
    m = xgb.XGBRegressor(objective="reg:absoluteerror", n_estimators=int(best[2] * 1.1), tree_method="hist",
                         enable_categorical=True, random_state=SEED, **best[1])
    m.fit(f[num + BASE_CAT], f["resid"])
    return val[BASELINE].to_numpy() + m.predict(v[num + BASE_CAT]), best[1]

def onehot(fit, others, num):
    cols = None; out = []
    for f in [fit] + others:
        x = pd.get_dummies(f[num + BASE_CAT].astype({c: str for c in BASE_CAT}), columns=BASE_CAT, dtype=float)
        cols = x.columns if cols is None else cols
        out.append(x.reindex(columns=cols, fill_value=0.0).fillna(x.median(numeric_only=True)).fillna(0.0))
    return out

ET_SPACE = {"min_samples_leaf": [5, 10, 20], "max_features": [0.3, 0.5, 1.0]}
def et_arm(fit, val, dev, hold, num):
    best = None
    xd, xh = onehot(dev, [hold], num)
    for p in sample(ET_SPACE, 6):
        m = ExtraTreesRegressor(n_estimators=400, random_state=SEED, n_jobs=-1, **p).fit(xd, dev["resid"])
        s = macro_mae(hold, hold[BASELINE].to_numpy() + m.predict(xh))
        if best is None or s < best[0]: best = (s, p)
    xf, xv = onehot(fit, [val], num)
    m = ExtraTreesRegressor(n_estimators=400, random_state=SEED, n_jobs=-1, **best[1]).fit(xf, fit["resid"])
    return val[BASELINE].to_numpy() + m.predict(xv), best[1]

def lad_arm(fit, val, dev, hold, num):
    def fit_lad(frame, alpha):
        x = onehot(frame, [], num)[0]; mu, sd = x.mean(), x.std().replace(0, 1)
        X = np.c_[np.ones(len(x)), ((x - mu) / sd).to_numpy()]; y = frame["resid"].to_numpy()
        P = np.eye(X.shape[1]); P[0, 0] = 0; w = np.ones(len(y))
        for _ in range(30):
            coef = np.linalg.solve(X.T @ (X * w[:, None]) + alpha * P, (X * w[:, None]).T @ y)
            w = 1 / np.maximum(np.abs(y - X @ coef), 1e-3)
        return x.columns, mu, sd, coef
    def predict(model, frame):
        cols, mu, sd, coef = model
        x = onehot(frame, [], num)[0].reindex(columns=cols, fill_value=0.0)
        return np.c_[np.ones(len(x)), ((x - mu) / sd).to_numpy()] @ coef
    alpha = min((0.1, 1, 10, 100), key=lambda a: macro_mae(hold, hold[BASELINE].to_numpy() + predict(fit_lad(dev, a), hold)))
    return val[BASELINE].to_numpy() + predict(fit_lad(fit, alpha), val), {"alpha": alpha}

def cb_rank_arm(fit, val, num, level_pred):
    def pool(frame, label=True):
        x = frame[num + BASE_CAT].copy()
        for c in BASE_CAT: x[c] = x[c].astype(str)
        return Pool(x, frame["resid"].to_numpy() if label else None, cat_features=BASE_CAT,
                    group_id=frame["channel_id"].astype("category").cat.codes.to_numpy())
    fit = fit.sort_values("channel_id", kind="stable"); val_sorted = val.sort_values("channel_id", kind="stable")
    params = {k: v for k, v in CB_PARAMS.items() if k != "iterations"}
    m = CatBoostRanker(loss_function="YetiRank", iterations=800, random_seed=SEED, verbose=False, allow_writing_files=False, **params)
    m.fit(pool(fit))
    score = pd.Series(m.predict(pool(val_sorted, label=False)), index=val_sorted.index).reindex(val.index)
    level = pd.Series(level_pred, index=val.index); out = level.copy()
    for _, idx in val.groupby("channel_id").groups.items():   # mesma distribuicao do CB_EMB, ordem do ranking
        out.loc[score.loc[idx].sort_values().index] = np.sort(level.loc[idx].to_numpy())
    return out.to_numpy()

## Execucao (rolling-origin)

In [ ]:
rows, chosen = [], []
t0 = time.time()
for origin, fit_idx, val_idx in folds(data):
    dev_idx, hold_idx = holdout(fit_idx)
    fit, val, dev, hold = with_pcs(fit_idx, [fit_idx, val_idx, dev_idx, hold_idx])
    num = BASE_NUM + EMB_COLS
    preds = {"B0": val[BASELINE].to_numpy()}
    preds["CB_EMB"] = cb_fit_predict(fit, val, num)
    for name, fn in (("LGBM", lgbm_arm), ("XGB", xgb_arm), ("ET", et_arm), ("LAD_EMB", lad_arm)):
        preds[name], p = fn(fit, val, dev, hold, num); chosen.append({"origin": origin, "arm": name, **p})
        print(f"{origin} {name} ({time.time() - t0:.0f}s)", flush=True)
    preds["ENS"] = (preds["CB_EMB"] + preds["LAD_EMB"]) / 2
    preds["CB_RANK"] = cb_rank_arm(fit, val, num, preds["CB_EMB"])
    preds["CB_KNN"] = cb_fit_predict(fit, val, num + KNN_COLS)
    w = 1.0 / fit.groupby("channel_id")["video_id"].transform("count").to_numpy()
    preds["CB_W"] = cb_fit_predict(fit, val, num, weight=w * len(w) / w.sum())
    print(f"{origin} completo ({time.time() - t0:.0f}s)", flush=True)
    frame = val[["video_id", "channel_id", "channel_name", "channel_type", TARGET, BASELINE]].copy(); frame["origin"] = origin
    for k, v in preds.items(): frame[f"pred_{k}"] = v
    rows.append(frame)
pred = pd.concat(rows, ignore_index=True)
MODELS = ["B0", "CB_EMB", "LGBM", "XGB", "ET", "LAD_EMB", "ENS", "CB_RANK", "CB_KNN", "CB_W"]
CANDIDATES = ["LGBM", "XGB", "ET", "ENS", "CB_RANK", "CB_KNN", "CB_W"]

## Metricas e regra pre-registrada (Holm)

In [ ]:
def pairwise(a, p):
    l, r = np.triu_indices(len(a), k=1); sa, sp = np.sign(a[l] - a[r]), np.sign(p[l] - p[r]); ok = sa != 0
    return float(np.where(sp[ok] == 0, 0.5, (sa[ok] == sp[ok]).astype(float)).mean()) if ok.any() else np.nan

recs = []
for (o, ch), g in pred.groupby(["origin", "channel_name"]):
    a = g[TARGET].to_numpy()
    for m in MODELS:
        p = g[f"pred_{m}"].to_numpy()
        recs.append({"origin": o, "channel_name": ch, "channel_type": g["channel_type"].iat[0], "model": m,
                     "mae": np.mean(np.abs(a - p)), "pair_fair": pairwise(a, p),
                     "spearman": spearmanr(a, p).correlation if np.ptp(p) > 0 and np.ptp(a) > 0 else 0.0})
pco = pd.DataFrame(recs)
pc = pco.groupby(["model", "channel_name", "channel_type"], as_index=False)[["mae", "pair_fair", "spearman"]].mean()
summary = pc.groupby("model")[["mae", "pair_fair", "spearman"]].mean().reindex(MODELS).round(4)
display(summary.sort_values("mae"))

def paired(cand, ref, metric):
    v = pc.pivot(index="channel_name", columns="model", values=metric); d = (v[cand] - v[ref]).dropna().to_numpy()
    r = np.random.default_rng(SEED); boot = r.choice(d, (10000, len(d))).mean(1); s = r.choice((-1, 1), (10000, len(d)))
    better = (d < 0) if metric == "mae" else (d > 0)
    return {"candidato": cand, "metrica": metric, "delta": float(d.mean()), "ic95_low": float(np.quantile(boot, .025)),
            "ic95_high": float(np.quantile(boot, .975)), "p": float((np.abs((s * d).mean(1)) >= abs(d.mean())).mean()),
            "canais_melhor": int(better.sum())}

tests = pd.DataFrame([paired(c, "CB_EMB", "pair_fair" if c == "CB_RANK" else "mae") for c in CANDIDATES])
order = tests["p"].sort_values().index; m = len(tests); adj, running = {}, 0.0
for rank, i in enumerate(order):                                   # Holm-Bonferroni
    running = max(running, min(1.0, (m - rank) * tests.loc[i, "p"])); adj[i] = running
tests["p_holm"] = pd.Series(adj)
tests["melhora"] = np.where(tests["metrica"] == "mae", tests["delta"] < 0, tests["delta"] > 0)
tests["aprovado"] = tests["melhora"] & (tests["p_holm"] < 0.05)
display(tests.round(4))
extra = pd.DataFrame([paired(c, "CB_EMB", "pair_fair" if c != "CB_RANK" else "mae") for c in CANDIDATES])
print("Leitura secundaria (nao decide):"); display(extra.round(4))
winner = tests.loc[tests["aprovado"]].sort_values("p_holm")["candidato"].tolist()
winner = winner[0] if winner else "CB_EMB"
print("Vencedor pela regra pre-registrada:", winner)

## Fase 2 · Optuna (origens 1-2 para escolher, origem 3 para avaliar)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
FOLDS = folds(data)
cache = {o: with_pcs(f, [f, v]) for o, f, v in FOLDS}
family = winner if winner in ("LGBM", "XGB") else "CB"
num_opt = BASE_NUM + EMB_COLS + (KNN_COLS if winner == "CB_KNN" else [])

def fit_eval(params, fit, val):
    if family == "CB":
        return cb_fit_predict(fit, val, num_opt, params=params)
    f, v = as_category(fit, [val], BASE_CAT)
    if family == "LGBM":
        m = lgb.LGBMRegressor(objective="l1", bagging_freq=1, random_state=SEED, verbose=-1, **params).fit(f[num_opt + BASE_CAT], f["resid"])
    else:
        m = xgb.XGBRegressor(objective="reg:absoluteerror", tree_method="hist", enable_categorical=True, random_state=SEED, **params).fit(f[num_opt + BASE_CAT], f["resid"])
    return val[BASELINE].to_numpy() + m.predict(v[num_opt + BASE_CAT])

def suggest(trial):
    if family == "CB":
        return {"depth": trial.suggest_int("depth", 4, 8), "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 30, log=True), "random_strength": trial.suggest_float("random_strength", 0.1, 5, log=True),
                "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 3), "iterations": trial.suggest_int("iterations", 300, 2000, step=100)}
    if family == "LGBM":
        return {"num_leaves": trial.suggest_int("num_leaves", 7, 63), "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 5, 60), "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
                "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0), "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10, log=True),
                "n_estimators": trial.suggest_int("n_estimators", 200, 2000, step=100)}
    return {"max_depth": trial.suggest_int("max_depth", 2, 8), "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
            "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True), "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0), "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20, log=True),
            "n_estimators": trial.suggest_int("n_estimators", 200, 2000, step=100)}

def objective(trial):
    params = suggest(trial)
    return float(np.mean([macro_mae(cache[o][1], fit_eval(params, *cache[o])) for o in ("origin_1", "origin_2")]))

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
t0 = time.time(); study.optimize(objective, n_trials=OPTUNA_TRIALS)
print(f"Optuna {family}: {OPTUNA_TRIALS} tentativas em {time.time() - t0:.0f}s | melhor MAE o1-o2 = {study.best_value:.4f}")
print(study.best_params)

fit3, val3 = cache["origin_3"]
ref_pred = pred.loc[pred["origin"] == "origin_3"].set_index("video_id")["pred_" + winner].reindex(val3["video_id"]).to_numpy()
opt_pred = fit_eval(study.best_params, fit3, val3)
o3 = val3[["video_id", "channel_name", TARGET]].assign(ref=ref_pred, opt=opt_pred)
per = o3.groupby("channel_name").apply(lambda g: pd.Series({"ref": np.mean(np.abs(g[TARGET] - g["ref"])), "opt": np.mean(np.abs(g[TARGET] - g["opt"]))}), include_groups=False)
d = (per["opt"] - per["ref"]).to_numpy(); r = np.random.default_rng(SEED); s = r.choice((-1, 1), (10000, len(d)))
optuna_result = {"family": family, "base": winner, "best_params": study.best_params, "mae_o12_best": study.best_value,
                 "origin_3_mae_ref": float(per["ref"].mean()), "origin_3_mae_optuna": float(per["opt"].mean()),
                 "delta": float(d.mean()), "p": float((np.abs((s * d).mean(1)) >= abs(d.mean())).mean()), "canais_melhor": int((d < 0).sum())}
optuna_result["aprovado"] = optuna_result["delta"] < 0 and optuna_result["p"] < 0.05
print(json.dumps(optuna_result, indent=1, default=float))

## PyCaret (exploratorio, nao decide)

In [ ]:
pycaret_result = "nao executado"
try:
    if os.environ.get("E12_SKIP_PYCARET"):
        raise RuntimeError("pulado por E12_SKIP_PYCARET")
    if importlib.util.find_spec("pycaret") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycaret"], check=True, timeout=600)
    from pycaret.regression import setup, compare_models, pull
    o3_fit, o3_val = cache["origin_3"]
    frame = pd.concat([o3_fit, o3_val]).reset_index(drop=True)[BASE_NUM + EMB_COLS + BASE_CAT + ["resid"]]
    n_fit = len(o3_fit)
    class OneSplit:                                  # split temporal fixo (sem CV aleatoria)
        def split(self, X, y=None, groups=None): yield np.arange(n_fit), np.arange(n_fit, len(X))
        def get_n_splits(self, *a, **k): return 1
    setup(data=frame, target="resid", categorical_features=BASE_CAT, fold_strategy=OneSplit(), data_split_shuffle=False,
          train_size=n_fit / len(frame), session_id=SEED, verbose=False, html=False)
    compare_models(sort="MAE", n_select=5, verbose=False)
    pycaret_result = pull().head(10)
    display(pycaret_result)
except Exception as error:
    pycaret_result = f"PyCaret indisponivel: {type(error).__name__}: {str(error)[:200]}"
    print(pycaret_result)

## Exportar

In [ ]:
pred.to_csv(os.path.join(OUT, "rolling_origin_predictions.csv"), index=False)
pco.to_csv(os.path.join(OUT, "per_channel_origin_metrics.csv"), index=False)
summary.to_csv(os.path.join(OUT, "summary.csv")); tests.to_csv(os.path.join(OUT, "holm_tests.csv"), index=False)
extra.to_csv(os.path.join(OUT, "secondary_tests.csv"), index=False)
pd.DataFrame(chosen).to_csv(os.path.join(OUT, "chosen_params.csv"), index=False)
report = {"experiment": "E-12", "winner_rule": winner, "summary": summary.reset_index().to_dict("records"),
          "holm": tests.to_dict("records"), "optuna": optuna_result,
          "pycaret": pycaret_result if isinstance(pycaret_result, str) else pycaret_result.reset_index().to_dict("records"),
          "n_trials": N_TRIALS, "optuna_trials": OPTUNA_TRIALS, "test_split": "fechado"}
json.dump(report, open(os.path.join(OUT, "report.json"), "w"), indent=2, ensure_ascii=False, default=str)
import shutil
archive = shutil.make_archive(os.path.join(WORK, "e12_outputs"), "zip", OUT); print(archive)
try:
    from google.colab import files; files.download(archive)
except ImportError:
    pass